# 10 — Sector Screen and Basket Construction

## Overview

This notebook documents the stock screening framework and basket construction tools.

### Screening Framework
The 5-pillar screen scores all 30 KSE 30 stocks:
1. **Performance** — 5yr TR CAGR, volatility, max drawdown, beta
2. **Fundamentals** — ROE
3. **Valuation** — P/E, P/B
4. **Dividend Quality** — Dividend yield, years without cut, growth
5. **Risk** — Beta, drawdown, volatility

### Portfolio Theory
- **Markowitz Efficient Frontier** — minimum variance portfolios at each return level
- **Diversification Curve** — portfolio volatility vs number of stocks
- **Recommendation Engine** — auto-selects top 8 stocks, weights for minimum variance

## 1. Load Modules

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from kse.screen import screen_all, print_screen_results
from kse.frontier import (
    load_stock_return_matrix,
    compute_efficient_frontier,
    compute_diversification_curve,
    get_recommended_portfolio
)
from kse.data_pipeline import load_stock_metrics

## 2. Stock Screen Results

In [ ]:
# Run the 5-pillar screen
print_screen_results()

### Screening Methodology

Each pillar is scored 1-5 within the stock's sector:
- **Performance:** CAGR > 20% → 5, > 15% → 4, > 10% → 3, > 5% → 2, else 1. Penalty for drawdown > 45%.
- **Fundamentals:** ROE > 30% → 5, > 25% → 4, > 20% → 3, > 15% → 2, else 1.
- **Valuation:** P/E < 4.5 → 5, < 5.5 → 4, < 7.0 → 3, < 9.0 → 2, else 1. Combined with P/B score.
- **Dividend Quality:** Yield > 8% → 5, > 6% → 4, > 4% → 3, > 2% → 2, else 1. Bonus for consistency and growth.
- **Risk:** Drawdown < 35% → 5, < 40% → 4, < 45% → 3, < 50% → 2, else 1. Adjusted by volatility.

Composite = average of all 5 pillars.

## 3. Basket Summary

In [ ]:
# Show stock-level data
stock_df = load_stock_metrics()
print(f"Total stocks in universe: {len(stock_df)}")
print()
print("Key metrics:")
print(f"  Avg dividend yield: {stock_df['dividend_yield'].mean():.1%}")
print(f"  Avg P/E:           {stock_df['pe'].mean():.1f}")
print(f"  Avg beta:          {stock_df['beta'].mean():.2f}")
print(f"  Avg max drawdown:  {stock_df['max_drawdown'].mean():.1%}")
print()
print("Sector breakdown:")
print(stock_df.groupby("sector").size().to_string())

## 4. Efficient Frontier

In [ ]:
# Load return matrix and compute frontier
returns_df, tickers = load_stock_return_matrix()
print(f"Loaded returns for {len(tickers)} stocks with {len(returns_df)} months of data")

if not returns_df.empty:
    frontier = compute_efficient_frontier(returns_df)
    
    # Plot the efficient frontier
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Efficient frontier (upper half only)
    min_ret = frontier["min_var"]["return"]
    eff = frontier["frontier"][frontier["frontier"]["return"] >= min_ret]
    ax.plot(eff["volatility"], eff["return"], color="#c96442", linewidth=2.5, label="Efficient frontier")
    
    # Individual stocks
    ss = frontier["stock_stats"]
    ax.scatter(ss["volatility"], ss["return"], color="#8a8580", s=30, alpha=0.7, label="Individual stocks")
    
    # Key portfolios
    ax.scatter(frontier["min_var"]["volatility"], frontier["min_var"]["return"], 
               color="#3f6fb5", s=100, marker="D", label="Min variance", zorder=5)
    ax.scatter(frontier["equal_weight"]["volatility"], frontier["equal_weight"]["return"], 
               color="#5a7a4a", s=100, marker="*", label="Equal weight", zorder=5)
    
    ax.set_xlabel("Annualized volatility (risk)")
    ax.set_ylabel("Annualized expected return")
    ax.set_title("Markowitz Efficient Frontier — KSE 30")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("../docs/efficient_frontier.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No return data available")

### Methodological Caveat

The efficient frontier is based on historical covariance with ~60 monthly observations for 30 stocks. The sample covariance matrix is poorly estimated, making the "optimal" portfolio unstable.

In practice, equal-weight portfolios often outperform optimized portfolios out-of-sample. Use this as an educational tool, not as investment advice.

## 5. Diversification Curve

In [ ]:
# Compute and plot diversification curve
if not returns_df.empty:
    div_curve = compute_diversification_curve(returns_df)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # P10-P90 band
    ax.fill_between(div_curve["n_stocks"], div_curve["p10_volatility"], 
                    div_curve["p90_volatility"], alpha=0.15, color="#c96442", label="P10-P90 range")
    
    # Average volatility
    ax.plot(div_curve["n_stocks"], div_curve["avg_volatility"], 
            color="#c96442", linewidth=2.5, marker="o", markersize=5, label="Average volatility")
    
    ax.set_xlabel("Number of stocks in portfolio")
    ax.set_ylabel("Annualized volatility")
    ax.set_title("Diversification Benefit — How Volatility Falls with More Stocks")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("../docs/diversification_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    # Key insight
    vol_1 = div_curve[div_curve["n_stocks"] == 1]["avg_volatility"].iloc[0]
    vol_10 = div_curve[div_curve["n_stocks"] == 10]["avg_volatility"].iloc[0]
    vol_30 = div_curve[div_curve["n_stocks"] == 30]["avg_volatility"].iloc[0]
    
    print(f"1 stock:  {vol_1:.1%} volatility")
    print(f"10 stocks: {vol_10:.1%} volatility ({(1-vol_10/vol_1):.0%} reduction)")
    print(f"30 stocks: {vol_30:.1%} volatility ({(1-vol_30/vol_1):.0%} reduction)")
    print()
    print("Most diversification benefit comes from the first 8-12 stocks.")
else:
    print("No return data available")

## 6. Recommendation Engine

In [ ]:
# Generate recommended portfolio
if not returns_df.empty:
    screen_df = screen_all()
    rec = get_recommended_portfolio(returns_df, screen_df)
    
    if rec:
        print("Recommended Portfolio (Minimum Variance):")
        print("=" * 60)
        print(f"Expected return: {rec['return']:.1%}")
        print(f"Volatility:      {rec['volatility']:.1%}")
        print()
        print("Composition:")
        for ticker, weight in zip(rec["tickers"], rec["weights"]):
            if weight > 0.001:
                print(f"  {ticker}: {weight:.1%}")
        print("=" * 60)
    else:
        print("Could not generate recommendation")
else:
    print("No return data available")

## 7. Key Findings

1. **The screen identifies quality.** FFC, MEBL, and POL consistently rank at the top — high dividends, strong ROE, reasonable valuation. Cement and technology rank lower — lower dividends, higher volatility.

2. **Diversification works, but with diminishing returns.** Most of the benefit comes from the first 8-12 stocks. Beyond that, the curve flattens.

3. **The efficient frontier is unstable.** With only 60 monthly observations for 30 stocks, the covariance matrix is poorly estimated. The "optimal" portfolio may not be optimal out-of-sample.

4. **Equal-weight is a strong benchmark.** In practice, equal-weight portfolios often outperform optimized portfolios. The minimum variance portfolio is a starting point, not a recommendation.

5. **Concentration risk is real.** A hand-picked basket has a deeper drawdown than the index. This is the price of trying to beat the index.